In [ ]:
%pip install -q --disable-pip-version-check semantic-link-labs

import sempy
import requests
import sempy_labs as labs
import sempy.fabric as fabric

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# Step 1: Download all labs

labs.import_notebook_from_web(overwrite=True, notebook_name="0. Create Environment", url="https://raw.githubusercontent.com/samudra-bandaranayake/SemanticModelDocumentation/refs/heads/main/0.%20Create%20Environment.ipynb")
labs.import_notebook_from_web(overwrite=True, notebook_name="0. Create Lake House", url="https://raw.githubusercontent.com/samudra-bandaranayake/SemanticModelDocumentation/refs/heads/main/0.%20Create%20Lake%20House.ipynb")
labs.import_notebook_from_web(overwrite=True, notebook_name="1. Run Notebooks", url="https://raw.githubusercontent.com/samudra-bandaranayake/SemanticModelDocumentation/refs/heads/main/1.%20Run%20Notebooks.ipynb")
labs.import_notebook_from_web(overwrite=True, notebook_name="2. Parameters", url="https://raw.githubusercontent.com/samudra-bandaranayake/SemanticModelDocumentation/refs/heads/main/2.%20Parameters.ipynb")
labs.import_notebook_from_web(overwrite=True, notebook_name="3. M Code Highlighter", url="https://raw.githubusercontent.com/samudra-bandaranayake/SemanticModelDocumentation/refs/heads/main/3.%20M%20Code%20Highlighter.ipynb")
labs.import_notebook_from_web(overwrite=True, notebook_name="4. Load Calc Dependency Data", url="https://raw.githubusercontent.com/samudra-bandaranayake/SemanticModelDocumentation/refs/heads/main/4.%20Load%20Calc%20Dependency%20Data.ipynb")
labs.import_notebook_from_web(overwrite=True, notebook_name="5. Data Transformation", url="https://raw.githubusercontent.com/samudra-bandaranayake/SemanticModelDocumentation/refs/heads/main/5.%20Data%20Transformation.ipynb")
labs.import_notebook_from_web(overwrite=True, notebook_name="6. Load Tables Columns", url="https://raw.githubusercontent.com/samudra-bandaranayake/SemanticModelDocumentation/refs/heads/main/6.%20Load%20Tables%20Columns.ipynb")
labs.import_notebook_from_web(overwrite=True, notebook_name="7. Load M Query", url="https://raw.githubusercontent.com/samudra-bandaranayake/SemanticModelDocumentation/refs/heads/main/7.%20Load%20M%20Query.ipynb")
labs.import_notebook_from_web(overwrite=True, notebook_name="8. Relationships", url="https://raw.githubusercontent.com/samudra-bandaranayake/SemanticModelDocumentation/refs/heads/main/8.%20Relationships.ipynb")
labs.import_notebook_from_web(overwrite=True, notebook_name="9. Extract Column List", url="https://raw.githubusercontent.com/samudra-bandaranayake/SemanticModelDocumentation/refs/heads/main/9.%20Extract%20Column%20List.ipynb")
labs.import_notebook_from_web(overwrite=True, notebook_name="10. Creating Semantic Model Diagram", url="https://raw.githubusercontent.com/samudra-bandaranayake/SemanticModelDocumentation/refs/heads/main/10.%20Creating%20Semantic%20Model%20Diagram.ipynb")
labs.import_notebook_from_web(overwrite=True, notebook_name="11. RLS", url="https://raw.githubusercontent.com/samudra-bandaranayake/SemanticModelDocumentation/refs/heads/main/11.%20RLS.ipynb")
labs.import_notebook_from_web(overwrite=True, notebook_name="12. Create Dim Database Table", url="https://raw.githubusercontent.com/samudra-bandaranayake/SemanticModelDocumentation/refs/heads/main/12.%20Create%20Dim%20Database%20Table.ipynb")

print('All Notebooks downloaded successfully')

client = fabric.FabricRestClient()
workspace_id = fabric.resolve_workspace_id()

# Create folder or get existing one
try:
    folder_response = client.post(
        f"/v1/workspaces/{workspace_id}/folders",
        json={"displayName": "Notebooks"}
    )
    folder_id = folder_response.json()["id"]
    print(f"Created 'Notebooks' folder: {folder_id}")

except Exception as e:
    if "409" in str(e) or "FolderDisplayNameAlreadyInUse" in str(e):
        print("Folder already exists, finding it...")
        folders = client.get(f"/v1/workspaces/{workspace_id}/folders").json()
        folder_id = next(f["id"] for f in folders["value"] if f["displayName"] == "Notebooks")
        print(f"Found existing 'Notebooks' folder: {folder_id}")
    else:
        raise

# Move numbered lab notebooks into the folder
items = client.get(f"/v1/workspaces/{workspace_id}/items?type=Notebook").json()
for item in items["value"]:
    if item["displayName"][:1].isdigit():
        move_resp = client.post(
            f"/v1/workspaces/{workspace_id}/items/{item['id']}/move",
            json={"targetFolderId": folder_id}
        )
        status = "moved" if move_resp.status_code == 200 else f"error {move_resp.status_code}"
        print(f"  {status}: {item['displayName']}")

## Configure Spark defaults
workspace_id = sempy.fabric.get_workspace_id()
client = fabric.FabricRestClient()

body = {
    "automaticLog": {
        "enabled": False
    },
    "highConcurrency": {
        "notebookInteractiveRunEnabled": True,
        "notebookPipelineRunEnabled": True
    },
    "pool": {
        "customizeComputeEnabled": False,
        "defaultPool": {
            "name": "Starter Pool",
            "type": "Workspace"
        },
        "starterPool": {
            "maxNodeCount": 1,
            "maxExecutors": 1
        }
    },
    "environment": {
        "runtimeVersion": "1.3"
    },
    "job": {
        "conservativeJobAdmissionEnabled": False,
        "sessionTimeoutInMinutes": 30
    }
}

response = client.patch(path_or_url=f"/v1/workspaces/{workspace_id}/spark/settings", json=body)
print(response)

labs.set_workspace_default_storage_format(storage_format="Large")